In [39]:
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline
import sklearn
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

DATA

In [40]:
corpus = [
    "góp gió gặt bão",
    "có làm mới có ăn",
    "đất lành chim đậu",
    "ăn cháo đá bát",
    "gậy ông đập lưng ông",
    "qua cầu rút ván"
]

n_doc = len(corpus)

labels = [1, 1, 1, 0, 0, 0] # 1: positive - 0: negative

cate_2_label = {
    "positive": 1,
    "negative": 0
}

In [ ]:
def label_2_cate(labels):
    key_list = list(cate_2_label.keys())
    value_list = list(cate_2_label.values())

    position = [value_list.index(label) for label in labels]
    print(position)
    return np.array(key_list)[position]

In [42]:
x_data = np.array(corpus)
print(x_data)
y_data = np.array(labels)
print(y_data)

['góp gió gặt bão' 'có làm mới có ăn' 'đất lành chim đậu' 'ăn cháo đá bát'
 'gậy ông đập lưng ông' 'qua cầu rút ván']
[1 1 1 0 0 0]


CONVERT TEXT TO VECTOR BY USING TF-IDF TRANSFORMATION

In [43]:
def calculate_tfidf(x_vectorized):
    tf = np.log(x_vectorized + 1)
    df = np.sum(x_vectorized, axis=0)
    idf = np.log((n_doc + 1) / (df + 1)) + 1
    tfidf = tf * idf

    return idf, tf, tfidf

def compute_norm(tfidf_vec):
    norm = np.linalg.norm(tfidf_vec, axis = 1)
    n_doc = tfidf_vec.shape[0]
    for i in range(n_doc):
        tfidf_vec[i] /= norm[i]

In [44]:
vectorizer = CountVectorizer()
x_vectorized = vectorizer.fit_transform(corpus).toarray()
print(f"vocab: {vectorizer.get_feature_names_out()}")

vocab: ['bát' 'bão' 'chim' 'cháo' 'có' 'cầu' 'gió' 'góp' 'gậy' 'gặt' 'làm' 'lành'
 'lưng' 'mới' 'qua' 'rút' 'ván' 'ông' 'ăn' 'đá' 'đất' 'đập' 'đậu']


In [45]:
x_idf, x_tf, x_tfidf = calculate_tfidf(x_vectorized)
print(f"x_idf: {x_idf}")
print(f"x_tf: {x_tf}")
print(f"x_tfidf: {x_tfidf}")

x_idf: [2.25276297 2.25276297 2.25276297 2.25276297 1.84729786 2.25276297
 2.25276297 2.25276297 2.25276297 2.25276297 2.25276297 2.25276297
 2.25276297 2.25276297 2.25276297 2.25276297 2.25276297 1.84729786
 1.84729786 2.25276297 2.25276297 2.25276297 2.25276297]
x_tf: [[0.         0.69314718 0.         0.         0.         0.
  0.69314718 0.69314718 0.         0.69314718 0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         1.09861229 0.
  0.         0.         0.         0.         0.69314718 0.
  0.         0.69314718 0.         0.         0.         0.
  0.69314718 0.         0.         0.         0.        ]
 [0.         0.         0.69314718 0.         0.         0.
  0.         0.         0.         0.         0.         0.69314718
  0.         0.         0.         0.         0.         0.
  0.         0.         0.69314718 0.         0.69314718]
 [0.693

NORMALIZE TF-IDF VALUES BY L2  NORM

In [46]:
compute_norm(x_tfidf)

In [47]:
# Training model
knn_cls = KNeighborsClassifier(n_neighbors=3)
knn_cls.fit(x_tfidf, y_data)
preds = knn_cls.predict(x_tfidf)
print(preds)

[1 0 1 1 1 1]


USING PIPELINE

In [ ]:
text_clf_model = Pipeline([('vect', CountVectorizer()),
                            ('tfidf', TfidfTransformer()),
                            ('clf', KNeighborsClassifier(n_neighbors=1)),
                        ])

text_clf_model.fit(X, y)

preds = text_clf_model.predict(X)
print(preds)

[1 1 1 0 0 0]


In [53]:
test_text = np.array(["không làm cạp đất mà ăn"])
test_vec = vectorizer.transform(test_text).toarray()
print(test_vec)

[[0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 0]]


In [54]:
test_tf = np.log(test_vec + 1)
test_tfidf = test_tf * x_idf

print(test_tfidf)

[[0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        1.5614963 0.        0.        0.
  0.        0.        0.        0.        1.2804493 0.        1.5614963
  0.        0.       ]]


In [55]:
compute_norm(test_tfidf)

In [60]:
pred = knn_cls.predict(test_tfidf)

print(label_2_cate(pred))

['positive']
